# Qwen3.5-9B: explicit free-runtime smoke
No inference has been performed by creating this notebook. Use an interactive **free** GPU runtime only after authorization. Sign in yourself; do not purchase compute, submit identity documents, mount Drive, expose Ollama or upload private/user data. If a free GPU is unavailable, stop. Upload only the synthetic bundle produced by `portable bundle`; paste its independently obtained SHA-256 below. Clear outputs before sharing this notebook.

In [ ]:
APPROVED_SYNTHETIC_TRANSFER = False
EXPECTED_SHA256 = "PASTE_BUNDLE_SHA256"
if not APPROVED_SYNTHETIC_TRANSFER or len(EXPECTED_SHA256) != 64:
    raise PermissionError("Obtain transfer permission and paste bundle hash first")
import hashlib
import io
import json
import os
import pathlib
import sys
import zipfile

from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one approved bundle")
data = next(iter(uploaded.values()))
if hashlib.sha256(data).hexdigest() != EXPECTED_SHA256:
    raise ValueError("Bundle hash mismatch")
root = pathlib.Path("/content/qwen-live")
root.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(data)) as archive:
    manifest = json.loads(archive.read("BUNDLE-MANIFEST.json"))
    if set(archive.namelist()) != set(manifest["sha256"]) | {"BUNDLE-MANIFEST.json"}:
        raise ValueError("Unexpected bundle inventory")
    for name, digest in manifest["sha256"].items():
        target = (root / name).resolve()
        if not target.is_relative_to(root) or target == root:
            raise ValueError("Invalid archive path")
        content = archive.read(name)
        if hashlib.sha256(content).hexdigest() != digest:
            raise ValueError("File hash mismatch")
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as stream:
            stream.write(content)
os.chdir(root)
sys.path.insert(0, str(root))
print("Committed synthetic bundle revision:", manifest["revision"])

In [ ]:
from experiments.qwen.portable import resources

preflight = resources()
print(json.dumps(preflight, indent=2))
if not preflight["qwen35_9b_gpu_candidate"]:
    raise ValueError("GPU/RAM/disk gate failed; do not download weights")

In [ ]:
APPROVED_RUNTIME_AND_DEPENDENCIES = False
APPROVED_QWEN35_9B_WEIGHTS = False
FREE_RUNTIME_CONFIRMED_IN_UI = False
if not all(
    [
        APPROVED_SYNTHETIC_TRANSFER,
        APPROVED_RUNTIME_AND_DEPENDENCIES,
        APPROVED_QWEN35_9B_WEIGHTS,
        FREE_RUNTIME_CONFIRMED_IN_UI,
    ]
):
    raise PermissionError("Separate permissions and free runtime required")
import subprocess

subprocess.run(
    [
        sys.executable,
        "-B",
        "-m",
        "experiments.qwen.bootstrap_cloud",
        "--approved-runtime",
        "--approved-weights",
        "--approved-synthetic-transfer",
        "--free-runtime-confirmed",
    ],
    check=True,
)

In [ ]:
# Export only task reports, never downloaded weights/runtime/account files.
import shutil

report_root = root / ".local/qwen-runs"
if not report_root.is_dir():
    raise ValueError("No report directory; inspect preflight/bootstrap error")
archive = shutil.make_archive("/content/qwen-live-results", "zip", report_root)
files.download(archive)

Inspect `diagnostics.json`, raw responses and prompt/model locks independently of `expert-review.json`. A technical PASS is not semantic approval. Model timeouts stop the sequence. The full F2 real-model test additionally needs the pinned F2 source plus an isolated PostgreSQL environment; this notebook does not claim to execute that path. Qwen3-4B is optional baseline and is not downloaded automatically.